# NDR - Notebook 13: Unlearn -> RELEARN (phuc hoi streak that)

## Vi sao (bang chung + literature)
- 09c: val 0.968, retention 0.72 NHUNG streak_miss_rate=0.28, pred_conf 0.68->0.39. Dan ap tot, PHUC HOI kem.
- Literature: 'Unified Neural Backdoor Removal via Unlearning AND Relearning' (2024); 'Impossibility of Retrain Equivalence' (2025) -> muc tieu behavior-matching.

## Thay doi so voi 09c
- Them nhanh RELEARN: dan synthetic streak (nhan DUNG 100%), ep student logits CAO tai vung streak -> keo conf streak that len, giam miss.
- Giu suppress + distill + cosine-anchor. Hop le co-author: weight that, khong pseudo-label tu model doc.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 656.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
# --- Confidence-fidelity diagnostic (proxy gan voi maCADD/clean model) ---
# maCADD phat theo |clean_score - depois_score| (A=10 bat doi xung). Firing/retention
# chi do CO/KHONG detect; cell nay do do KHOP CONFIDENCE voi clean model tren streak that,
# va residual confidence tren poison. Gan hon voi hanh vi top-1 (recovery + calibration).
def conf_fidelity(predictor, val, clean_ref, tta=False, floor=SUBMIT_THRESH):
    gaps=[]; pred_on_streak=[]; ref_on_streak=[]; missed=0; total_streak=0
    pois_res=[]; total_pois=0
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor,item['img']) if tta else predict_image(predictor,item['img']))
        keep = s>floor; b=(b[keep] if len(b) else b); s=(s[keep] if len(b) else s)
        rb,rs = clean_ref[str(i)]
        for k in range(len(rb)):
            total_streak+=1
            if len(b):
                ious=iou_matrix(rb[k:k+1],b)[0]; j=int(ious.argmax())
                if ious[j]>=MATCH_IOU:
                    gaps.append(abs(float(rs[k])-float(s[j]))); ref_on_streak.append(float(rs[k])); pred_on_streak.append(float(s[j])); continue
            gaps.append(float(rs[k])); missed+=1   # streak bi bo lo -> gap = toan bo conf ref (dong gop FN)
        pb=np.array(item['poison_boxes'],np.float32).reshape(-1,4)
        for k in range(len(pb)):
            total_pois+=1
            if len(b):
                ious=iou_matrix(pb[k:k+1],b)[0]
                if ious.max()>=MATCH_IOU: pois_res.append(float(s[int(ious.argmax())]))
    return dict(
        streak_conf_gap=float(np.mean(gaps)) if gaps else 0.0,
        streak_miss_rate=missed/max(total_streak,1),
        pred_conf_on_streak=float(np.mean(pred_on_streak)) if pred_on_streak else 0.0,
        ref_conf_on_streak=float(np.mean(ref_on_streak)) if ref_on_streak else 0.0,
        poison_residual_conf=float(np.mean(pois_res)) if pois_res else 0.0,
        poison_fire_frac=len(pois_res)/max(total_pois,1))

base_fid = conf_fidelity(poisoned_predictor, VAL, CLEAN_REF)
print('POISONED conf-fidelity:', {k:round(v,4) for k,v in base_fid.items()})
print('-> Muc tieu: streak_conf_gap THAP (khop conf clean model tren streak that),')
print('   poison_residual_conf ~0 (khong con FP poison). Ca hai giam = gan hanh vi top-1.')


POISONED conf-fidelity: {'streak_conf_gap': 0.0194, 'streak_miss_rate': 0.0, 'pred_conf_on_streak': 0.6822, 'ref_conf_on_streak': 0.6627, 'poison_residual_conf': 0.5937, 'poison_fire_frac': 0.9693}
-> Muc tieu: streak_conf_gap THAP (khop conf clean model tren streak that),
   poison_residual_conf ~0 (khong con FP poison). Ca hai giam = gan hanh vi top-1.


In [9]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. Baseline + neo LB

In [10]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
LB_ANCHOR_MACADD = 1.076   # local val_macadd của p05 (=LB 261.2)
print('POISONED baseline:', {k: round(v,4) for k,v in base_stats.items()})
print('Local neo (neo NB4 = LB258.7): val_macadd =', LB_ANCHOR_MACADD)
print('-> Chỉ nộp NB4 nếu val_macadd < %.3f VÀ retention >= 0.6' % LB_ANCHOR_MACADD)

POISONED baseline: {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
Local neo (neo NB4 = LB258.7): val_macadd = 1.076
-> Chỉ nộp NB4 nếu val_macadd < 1.076 VÀ retention >= 0.6


## 8. Retain-thật từ ảnh test (dùng cho distillation)

In [11]:
IMG_STORE = {}
def build_real_retain(n_test=250, retain_conf=0.4, n_poison_imgs=20, add_synth=True, seed=SEED):
    '''Retain set IN-DISTRIBUTION: nền ảnh test thật + pseudo-GT streak từ model gốc,
    TĂNG mật độ bằng synthetic streak có nhãn chuẩn (chống retain thưa gây under-train).
    Poison patch dán (không nhãn) = forget signal.'''
    rng = np.random.default_rng(seed); records = []; npos = 0; nposimg = 0
    pick = list(range(len(test_files))); rng.shuffle(pick); pick = pick[:n_test]
    for k in pick:
        p = test_files[k]; img = load_image(p); anns = []
        b, sc = predict_image(poisoned_predictor, img)
        rb = b[sc >= retain_conf] if len(b) else b
        for (x1,y1,x2,y2) in rb:
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
            x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[x1,y1,x2,y2],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        if add_synth:  # bơm thêm synthetic streak (nhãn CHẮC CHẮN đúng) lên nền test thật
            g = img[:,:,0].copy()
            for _ in range(rng.integers(1,4)):
                g, box = synth_streak_on(g, rng)
                x1,y1,x2,y2 = box
                if x2-x1>1 and y2-y1>1:
                    anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],
                                 'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
            img = np.repeat(g[:,:,None],3,axis=2)
        iid = 20_000_000 + int(k); IMG_STORE[iid] = img.astype(np.float32)
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    # ảnh poison-forget từ unlearn set
    for j in range(n_poison_imgs):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); anns=[]
        b, sc = predict_image(poisoned_predictor, img)
        pb = poison_boxes_by_stem.get(stem, np.zeros((0,4),np.float32))
        keep = sc >= retain_conf
        if len(b) and len(pb) and keep.any():
            keep = keep & (iou_matrix(b, pb).max(1) < POISON_OVERLAP_IOU)
        for (x1,y1,x2,y2) in (b[keep] if len(b) else b):
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        for _ in range(rng.integers(1,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem, ph, pw, rng)
            if spot: paste_patch(img, patch, spot[0], spot[1])
        iid = 21_000_000 + j; IMG_STORE[iid] = img.astype(np.float32); nposimg += 1
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    rng.shuffle(records)
    return records, npos, nposimg
IMG_STORE.clear()
_r,_np,_npi = build_real_retain(n_test=80, n_poison_imgs=10, seed=1)
print(f'probe retain-thật+synth: {len(_r)} ảnh | {_np} pseudo-GT streak ({_np/max(len(_r),1):.2f}/ảnh) | {_npi} ảnh poison-forget')
IMG_STORE.clear()

probe retain-thật+synth: 90 ảnh | 208 pseudo-GT streak (2.31/ảnh) | 10 ảnh poison-forget


## 9. Forward khả vi lấy cls logits (cho distillation)
Trả về list logits mỗi FPN level (chưa sigmoid). Teacher & student dùng chung hàm này.

### 8b. Detoxify: prune channel poison (cls+bbox) — RobustKD step 1

In [12]:
def _prune_subnet(model, seq, prune_frac, poison_xywh, tag):
    '''Prune channel phản hồi poison mạnh trong 1 subnet (cls hoặc bbox).'''
    convs=[m for m in seq if isinstance(m, nn.Conv2d)]
    stored={i:[] for i in range(len(convs))}; hooks=[]
    for i,ly in enumerate(convs):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts=[{'file_name':str(Path(UNLEARN_DIR)/f'{s}.png'),'image_id':int(s),'height':IMG_H,'width':IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im=load_image(d['file_name']); model([{'image':torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    total=0
    for li,layer in enumerate(convs):
        acts=stored[li]; C=acts[0].shape[1]; fg=torch.zeros(C); bg=torch.zeros(C); nf=nb_=0
        rng=np.random.default_rng(SEED)
        for act,d in zip(acts,dicts):
            a=act[0]; _,aH,aW=a.shape; sy,sx=aH/d['height'],aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'],[]):
                x1,y1=max(0,int(x*sx)),max(0,int(y*sy)); x2,y2=min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg+=a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw=max(1,int(16*sy)),max(1,int(16*sx)); ry,rx=rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg+=a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score=(fg-bg).numpy(); n_prune=max(1,int(len(score)*prune_frac)); bad=np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        total+=n_prune
    return total

def collect_and_prune(model, prune_frac, prune_box=True):
    model.eval(); total=0
    total += _prune_subnet(model, model.head.cls_subnet, prune_frac, poison_xywh, 'cls')
    if prune_box and hasattr(model.head,'bbox_subnet'):
        total += _prune_subnet(model, model.head.bbox_subnet, prune_frac, poison_xywh, 'box')
    return model, total

In [13]:
def cls_logits(model, x255, feats_out=None):
    '''x255: tensor [3,H,W] RGB 0-255. Trả list logits mỗi level; nếu feats_out là list, append FPN feats.'''
    nrm = ((x255 - model.pixel_mean) / model.pixel_std)[None]
    feats = model.backbone(nrm)
    fl = [feats[f] for f in model.head_in_features]
    if feats_out is not None: feats_out.extend([f for f in fl])
    logits, _ = model.head(fl)
    return logits

def load_teacher_detox(prune_frac=0.05, prune_box=True):
    '''RobustKD step1: teacher = poisoned model ĐÃ prune channel poison (detoxify).'''
    m = build_model(build_cfg(POISONED_WEIGHTS)); DetectionCheckpointer(m).load(POISONED_WEIGHTS)
    m.to(DEVICE)
    if prune_frac>0: m,_ = collect_and_prune(m, prune_frac, prune_box=prune_box)
    m.eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m
print('cls_logits + load_teacher_detox defined.')

cls_logits + load_teacher_detox defined.


## 10. Self-distillation trainer
- **Distill (giữ):** trên ảnh test SẠCH, student khớp logits teacher (MSE) → giữ hành vi clean, retention.
- **Suppress (quên):** dán poison patch lên ảnh, ép student logits tại vùng patch xuống thấp → forget poison.
- Backbone freeze, chỉ head học. λ_sup cân bằng 2 lực.

## 10. Trainer: suppress(quen) + distill(giu) + RELEARN(phuc hoi streak) + cosine

In [14]:
def paste_poison_get_region(img, rng):
    im = img.copy(); regions=[]
    for _ in range(rng.integers(1,3)):
        patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
        y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
        im[y0:y0+ph, x0:x0+pw] = patch; regions.append((x0,y0,x0+pw,y0+ph))
    return im, regions

def paste_streak_get_region(img, rng):
    g = img[:,:,0].copy(); regions=[]
    for _ in range(rng.integers(1,3)):
        g, box = synth_streak_on(g, rng)
        x1,y1,x2,y2 = [int(v) for v in box]
        if x2>x1 and y2>y1: regions.append((x1,y1,x2,y2))
    return np.repeat(g[:,:,None],3,axis=2), regions

def _region_mask_like(logit, regions, H, W):
    _,_,Hi,Wi = logit.shape; m = torch.zeros(1,1,Hi,Wi, device=logit.device)
    sy,sx = Hi/H, Wi/W
    for (x1,y1,x2,y2) in regions:
        a,b=int(y1*sy),max(int(y2*sy)+1,int(y1*sy)+1); cc,d=int(x1*sx),max(int(x2*sx)+1,int(x1*sx)+1)
        m[:,:,a:b,cc:d]=1.0
    return m

def train_distill(prune_teacher=0.05, n_test=250, lr=1e-4, iters=400, lam_sup=1.0,
                  lam_fvar=1.0, sup_ratio=0.35, lam_cos=0.5, lam_relearn=1.0,
                  relearn_ratio=0.30, lam_l2=0.0, seed=SEED, tag="r0"):
    out=WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    teacher = load_teacher_detox(prune_frac=prune_teacher)
    student = build_model(build_cfg(POISONED_WEIGHTS, output_dir=out))
    DetectionCheckpointer(student).load(POISONED_WEIGHTS); student.to(DEVICE)
    for n,p in student.named_parameters(): p.requires_grad = not n.startswith("backbone.")
    opt=torch.optim.Adam([p for p in student.parameters() if p.requires_grad], lr=lr)
    anchor={n:p.detach().clone() for n,p in student.named_parameters() if p.requires_grad}
    rng=np.random.default_rng(seed)
    pick=rng.choice(len(test_files), size=min(n_test,len(test_files)), replace=False)
    imgs=[load_image(test_files[int(k)]) for k in pick]
    student.train()
    for step in range(iters):
        img = imgs[int(rng.integers(len(imgs)))]
        r = rng.random(); tf=[]; sf=[]
        if r < relearn_ratio:
            im2, regions = paste_streak_get_region(img, rng)
            if not regions:
                xt=torch.as_tensor(img.transpose(2,0,1),dtype=torch.float32,device=DEVICE)
                with torch.no_grad(): t_log=cls_logits(teacher,xt,tf)
                s_log=cls_logits(student,xt,sf)
                loss=sum(((sl-tl)**2).mean() for tl,sl in zip(t_log,s_log))
            else:
                xt=torch.as_tensor(im2.transpose(2,0,1),dtype=torch.float32,device=DEVICE)
                with torch.no_grad(): t_log=cls_logits(teacher,xt,tf)
                s_log=cls_logits(student,xt,sf)
                loss=0.0
                for tl,sl in zip(t_log,s_log):
                    mask=_region_mask_like(sl, regions, IMG_H, IMG_W)
                    w=1.0+lam_relearn*mask
                    loss=loss+(((sl-tl)**2)*w).mean()
        elif r < relearn_ratio + sup_ratio:
            im2, regions = paste_poison_get_region(img, rng)
            xt = torch.as_tensor(im2.transpose(2,0,1), dtype=torch.float32, device=DEVICE)
            with torch.no_grad(): t_log = cls_logits(teacher, xt, tf)
            s_log = cls_logits(student, xt, sf)
            loss=0.0
            for tl,sl in zip(t_log,s_log):
                mask=_region_mask_like(sl, regions, IMG_H, IMG_W)
                loss=loss+((sl-tl)**2*(1-mask)).mean()
                loss=loss+lam_sup*(torch.sigmoid(sl)**2*mask).mean()
        else:
            xt = torch.as_tensor(img.transpose(2,0,1), dtype=torch.float32, device=DEVICE)
            with torch.no_grad(): t_log = cls_logits(teacher, xt, tf)
            s_log = cls_logits(student, xt, sf)
            loss=sum(((sl-tl)**2).mean() for tl,sl in zip(t_log,s_log))
        if lam_fvar>0 and tf and sf:
            fvar=sum((sfi-tfi).var() for sfi,tfi in zip(sf,tf))/len(sf); loss=loss+lam_fvar*fvar
        if lam_cos>0:
            cos=torch.tensor(0.0,device=DEVICE)
            for n,p in student.named_parameters():
                if p.requires_grad and n in anchor:
                    cos=cos+(1-torch.nn.functional.cosine_similarity(anchor[n].flatten(),p.flatten(),dim=0))
            loss=loss+lam_cos*cos
        if lam_l2>0:  # EWC-style L2 anchor: giu CA DO LON weight goc (khong chi huong nhu cos)
            l2=torch.tensor(0.0,device=DEVICE)  # -> giu calibration model goc -> robust tren anh that/private
            for n,p in student.named_parameters():
                if p.requires_grad and n in anchor:
                    l2=l2+((p-anchor[n])**2).sum()
            loss=loss+lam_l2*l2
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1)%150==0: print(f"  it{step+1}: loss={float(loss):.4f}")
    final=str(out/"model_final.pth"); torch.save({"model":student.state_dict()}, final)
    del teacher, student; torch.cuda.empty_cache()
    return final
print("train_distill + RELEARN defined.")

train_distill + RELEARN defined.


## 11. Sweep relearn - chon theo val_macadd + miss_rate

In [15]:
# SWEEP tai cau truc quanh r3 (ban tot nhat: retention 0.855). Muc tieu PRIVATE:
# - them lam_l2 (EWC-anchor) giu calibration model goc -> robust tren anh that.
# - GIAM iters (600->300) de bot lech calibration do fine-tune lau tren synthetic.
# - thien RETENTION cao: relearn_ratio cao hon, lam_sup thap hon.
SWEEP=[
    dict(prune_teacher=0.03, n_test=400, lr=1e-4, iters=300, lam_sup=0.5, lam_fvar=2.0, sup_ratio=0.30, lam_cos=0.5, lam_relearn=1.0, relearn_ratio=0.25, lam_l2=0.0,  seed=13, tag="r3"),
    dict(prune_teacher=0.03, n_test=400, lr=1e-4, iters=300, lam_sup=0.4, lam_fvar=2.0, sup_ratio=0.25, lam_cos=0.5, lam_relearn=1.5, relearn_ratio=0.35, lam_l2=1e-4, seed=13, tag="r3l2"),
    dict(prune_teacher=0.03, n_test=400, lr=1e-4, iters=250, lam_sup=0.4, lam_fvar=2.0, sup_ratio=0.25, lam_cos=0.5, lam_relearn=1.5, relearn_ratio=0.40, lam_l2=3e-4, seed=7,  tag="rret"),
    dict(prune_teacher=0.05, n_test=400, lr=1e-4, iters=300, lam_sup=0.5, lam_fvar=2.0, sup_ratio=0.30, lam_cos=0.5, lam_relearn=1.0, relearn_ratio=0.30, lam_l2=1e-4, seed=42, tag="r0l2"),
]
sweep_results=[]
for spec in SWEEP:
    w=train_distill(**spec)
    pr=build_predictor(w, CAND_THRESH); stats=eval_model_on_val(pr, VAL)
    fid=conf_fidelity(pr, VAL, CLEAN_REF); del pr; torch.cuda.empty_cache()
    sweep_results.append({**spec,"weights":w,**stats,**{"fid_"+k:v for k,v in fid.items()}})
    print(f"{spec['tag']}: firing={stats['poison_firing_rate']:.3f} retention={stats['streak_retention']:.3f} val={stats['val_macadd']:.4f}")
    print(f"    conf_gap={fid['streak_conf_gap']:.4f} miss={fid['streak_miss_rate']:.3f} pred_conf={fid['pred_conf_on_streak']:.3f} poison_resid={fid['poison_residual_conf']:.4f}")
sweep_df=pd.DataFrame(sweep_results).sort_values("val_macadd")
print("\n", sweep_df[["tag","lam_relearn","relearn_ratio","poison_firing_rate","streak_retention","val_macadd","fid_streak_miss_rate","fid_pred_conf_on_streak"]].to_string(index=False))
print(f"\nneo 09c: val=0.9683 retention=0.72 miss=0.277 pred_conf=0.392")
print(f"POISONED: val={base_stats['val_macadd']:.4f}")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


  it150: loss=0.0063
  it300: loss=0.0038


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


r3: firing=0.656 retention=0.843 val=1.0353
    conf_gap=0.3105 miss=0.157 pred_conf=0.431 poison_resid=0.3054


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

rret: firing=0.534 retention=0.761 val=1.0169
    conf_gap=0.3498 miss=0.239 pred_conf=0.421 poison_resid=0.2892


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


  it150: loss=0.0096
  it300: loss=0.0056


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


r0l2: firing=0.141 retention=0.528 val=1.0070
    conf_gap=0.4748 miss=0.472 pred_conf=0.360 poison_resid=0.2484

  tag  lam_relearn  relearn_ratio  poison_firing_rate  streak_retention  val_macadd  fid_streak_miss_rate  fid_pred_conf_on_streak
r0l2          1.0           0.30            0.141104          0.528302    1.007040              0.471698                 0.359651
rret          1.5           0.40            0.533742          0.761006    1.016856              0.238994                 0.421448
  r3          1.0           0.25            0.656442          0.842767    1.035315              0.157233                 0.430989
r3l2          1.5           0.35            0.668712          0.842767    1.040202              0.157233                 0.439318

neo 09c: val=0.9683 retention=0.72 miss=0.277 pred_conf=0.392
POISONED: val=1.4549


## 12. Chon cau hinh: uu tien retention>=0.65 & miss thap, val<neo

In [16]:
NEO=1.076
# MUC TIEU: PRIVATE cao. FN (miss streak) bi phat FULL-weight va la thu lech nhieu nhat
# giua public/private -> uu tien RETENTION cao (robust) thay vi chay theo dinh val local.
qualified=[r for r in sweep_results if r["val_macadd"]<NEO]
pool=qualified if qualified else sweep_results
# chon: retention cao nhat truoc (lam tron 2 so de bot nhieu), val_macadd thap lam tie-break
best=max(pool, key=lambda r:(round(r["streak_retention"],2), -r["val_macadd"]))
# SOUP chi tu cac ban RETENTION CAO (>=0.65) de khong pha loang recall; giam variance -> robust private
soup_src=[r for r in sweep_results if r["streak_retention"]>=0.65 and r["val_macadd"]<NEO]
if len(soup_src)>=2:
    sds=[torch.load(r["weights"],map_location="cpu")["model"] for r in soup_src]
    avg={k:sum(sd[k].float() for sd in sds)/len(sds) for k in sds[0]}
    sw=str(WORK/"soup.pth"); torch.save({"model":avg}, sw)
    pr=build_predictor(sw,CAND_THRESH); ss=eval_model_on_val(pr,VAL); sfd=conf_fidelity(pr,VAL,CLEAN_REF); del pr; torch.cuda.empty_cache()
    print("SOUP:", {k:round(v,4) for k,v in ss.items()}, "| miss", round(sfd["streak_miss_rate"],3))
    # uu tien soup neu giu retention>=0.65 VA val trong sai so 5% cua best (on dinh > dinh local)
    if ss["streak_retention"]>=0.65 and ss["val_macadd"]<=best["val_macadd"]*1.05:
        best={"tag":"soup","weights":sw,**ss,**{"fid_"+k:v for k,v in sfd.items()}}
FINAL_WEIGHTS=best["weights"]; FINAL_STATS={k:best[k] for k in ("poison_firing_rate","streak_retention","val_macadd")}
print("CHON:", best["tag"], {k:round(float(v),4) for k,v in FINAL_STATS.items()}, "| miss", round(best.get("fid_streak_miss_rate",-1),3))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


SOUP: {'poison_firing_rate': 0.5828, 'streak_retention': 0.8239, 'val_macadd': 1.0105} | miss 0.176
CHON: soup {'poison_firing_rate': 0.5828, 'streak_retention': 0.8239, 'val_macadd': 1.0105} | miss 0.176


## 16. Filter kieu top-1: embedding-sim + length + conf floor
Tai dung dung pipeline cua top-1 (giai ma tu Debris Best CSVs):
`simple-ft model -> score dets -> drop sim_max>=T -> length-rescue -> conf floor`.
Bai hoc v1->v20 da bake vao:
- NB20 fail vi embed bang **backbone model DOC** (separation 0.077). O day dung encoder **SACH** (ImageNet ResNet50) + van so voi backbone doc, tu chon theo separation.
- Top-1 co: corr(sim,conf)=0.074 (embedding TRUC GIAO conf), corr(sim,length)=-0.433 (box DAI -> sim thap -> streak that).
- Top-1 T~0.955-0.96, mat do ~0.2 det/img. Ta neo T theo top-1, KHONG calibrate mu tren VAL synthetic (thu da lua NB14/16/20).
- FINAL_WEIGHTS (de-poison ~247) dong vai tro 'simple-ft model'. Filter chi RESCORE, khong sua weight.
- An toan: ghi 2 submission. submission.csv = ban 247 proven (fallback). submission_filtered.csv = ban top-1 style (chi vuot khi VAL xac nhan retention>=0.85 & val thap hon).

In [17]:
## 16a. Encoder shootout: nhieu encoder + feature-level + crop-size
import torchvision
IMAGENET_MEAN = torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)

# ResNet50 ImageNet + hook lay nhieu tang (nong = texture, sau = semantic)
_rn = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
_rn.eval().to(DEVICE)
for p in _rn.parameters(): p.requires_grad_(False)
_feats = {}
def _mkhook(name):
    def h(m,i,o): _feats[name] = o
    return h
_rn.layer1.register_forward_hook(_mkhook("l1"))
_rn.layer2.register_forward_hook(_mkhook("l2"))
_rn.layer3.register_forward_hook(_mkhook("l3"))
_rn.layer4.register_forward_hook(_mkhook("l4"))

def _prep(crops, size):
    xs=[]
    for c in crops:
        c01 = np.clip(np.asarray(c,np.float32)/255.0,0,1)
        if c01.ndim==2: c01 = np.repeat(c01[:,:,None],3,2)
        xs.append(torch.as_tensor(cv2.resize(c01,(size,size)).transpose(2,0,1),dtype=torch.float32))
    return (torch.stack(xs).to(DEVICE)-IMAGENET_MEAN)/IMAGENET_STD

def _mk_rn(level, size):
    @torch.no_grad()
    def fn(crops):
        _ = _rn(_prep(crops,size))
        return torch.nn.functional.normalize(_feats[level].mean(dim=[2,3]), dim=1)
    return fn

# CLIP (neu co) - thuong tach semantic tot hon ImageNet thuan
_clip_fn = None
try:
    import open_clip
    _clipm,_,_ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    _clipm.eval().to(DEVICE)
    for p in _clipm.parameters(): p.requires_grad_(False)
    _CM = torch.tensor([0.4814,0.4578,0.4082],device=DEVICE).view(1,3,1,1)
    _CS = torch.tensor([0.2686,0.2613,0.2758],device=DEVICE).view(1,3,1,1)
    @torch.no_grad()
    def _clip_fn(crops):
        xs=[]
        for c in crops:
            c01=np.clip(np.asarray(c,np.float32)/255.0,0,1)
            if c01.ndim==2: c01=np.repeat(c01[:,:,None],3,2)
            xs.append(torch.as_tensor(cv2.resize(c01,(224,224)).transpose(2,0,1),dtype=torch.float32))
        x=(torch.stack(xs).to(DEVICE)-_CM)/_CS
        return torch.nn.functional.normalize(_clipm.encode_image(x).float(), dim=1)
    print("CLIP ViT-B/32 loaded")
except Exception as e:
    print("CLIP unavailable (bo qua):", str(e)[:80])

# backbone model DOC (doi chung, giong NB20)
_pm = poisoned_predictor.model
_PM = _pm.pixel_mean.view(1,3,1,1); _PS = _pm.pixel_std.view(1,3,1,1)
@torch.no_grad()
def embed_poisoned(crops):
    xs=[torch.as_tensor(cv2.resize(np.asarray(c,np.float32),(96,96)).transpose(2,0,1),dtype=torch.float32) for c in crops]
    x=(torch.stack(xs).to(DEVICE)-_PM)/_PS
    fd=_pm.backbone.bottom_up(x); f=fd[sorted(fd)[-1]].mean(dim=[2,3])
    return torch.nn.functional.normalize(f,dim=1)

ENC_CANDS = {
    "rn_l2_64":  _mk_rn("l2",64),   "rn_l2_128": _mk_rn("l2",128),
    "rn_l3_64":  _mk_rn("l3",64),   "rn_l3_128": _mk_rn("l3",128),
    "rn_l4_64":  _mk_rn("l4",64),   "rn_l4_128": _mk_rn("l4",128),
    "poisoned":  embed_poisoned,
}
if _clip_fn is not None: ENC_CANDS["clip_vitb32"] = _clip_fn
print("encoder candidates:", list(ENC_CANDS))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 169MB/s]


CLIP unavailable (bo qua): No module named 'open_clip'
encoder candidates: ['rn_l2_64', 'rn_l2_128', 'rn_l3_64', 'rn_l3_128', 'rn_l4_64', 'rn_l4_128', 'poisoned']


In [18]:
## 16b. Poison bank + shootout separation tren VAL -> chon encoder theo Cohen's d
def _clip_box(b):
    x1,y1,x2,y2=[int(round(v)) for v in b]; return max(0,x1),max(0,y1),min(IMG_W,x2),min(IMG_H,y2)
def _crops_from(boxes,img):
    crops=[];idx=[]
    for i in range(len(boxes)):
        x1,y1,x2,y2=_clip_box(boxes[i])
        if x2-x1>=2 and y2-y1>=2: crops.append(img[y1:y2,x1:x2]); idx.append(i)
    return crops,idx
def sim_max_for(boxes,img,embed_fn,bank):
    out=np.zeros(len(boxes),np.float32)
    if len(boxes)==0: return out
    crops,idx=_crops_from(boxes,img)
    if crops:
        sims=(embed_fn(crops)@bank.t()).max(dim=1).values.cpu().numpy()
        for k,i in enumerate(idx): out[i]=float(sims[k])
    return out
def length_of(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    return np.maximum(boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1]).astype(np.float32)

final_pr = build_predictor(FINAL_WEIGHTS, CAND_THRESH)
VAL_CANDS=[predict_image(final_pr, VAL[i]["img"]) for i in range(len(VAL))]
def build_bank(embed_fn): return embed_fn(poison_patches)

def separation(embed_fn, bank):
    ps=[]; ss=[]
    for i,item in enumerate(VAL):
        b0,_=VAL_CANDS[i]
        if len(b0)==0: continue
        sim=sim_max_for(b0, VAL[i]["img"], embed_fn, bank)
        sb=np.array(item["streak_boxes"],np.float32).reshape(-1,4); pb=np.array(item["poison_boxes"],np.float32).reshape(-1,4)
        for j in range(len(b0)):
            box=b0[j:j+1]
            if len(pb) and iou_matrix(box,pb).max()>=MATCH_IOU: ps.append(sim[j])
            elif len(sb) and iou_matrix(box,sb).max()>=MATCH_IOU: ss.append(sim[j])
    ps=np.array(ps); ss=np.array(ss)
    pm=float(ps.mean()) if len(ps) else 0.0
    sm=float(ss.mean()) if len(ss) else 0.0
    pooled=np.sqrt(((ps.var() if len(ps)>1 else 0.0)+(ss.var() if len(ss)>1 else 0.0))/2)+1e-9
    return pm,sm,len(ps),len(ss),(pm-sm)/pooled

ENC={}
for name,fn in ENC_CANDS.items():
    try:
        bank=build_bank(fn); pm,sm,npo,nst,d=separation(fn,bank)
    except Exception as e:
        print(f"{name:12s}: FAILED {str(e)[:60]}"); continue
    ENC[name]=dict(fn=fn,bank=bank,sep=pm-sm,pmean=pm,smean=sm,d=d)
    print(f"{name:12s}: poison={pm:.3f} streak={sm:.3f} sep={pm-sm:+.3f} cohens_d={d:+.3f} (np={npo} ns={nst})")
# chon theo Cohen's d (chuan hoa theo std -> so sanh cong bang giua cac encoder khac thang cosine)
BEST_ENC=max(ENC, key=lambda k:ENC[k]["d"])
EMBED_FN=ENC[BEST_ENC]["fn"]; POIS_BANK=ENC[BEST_ENC]["bank"]; SEP=ENC[BEST_ENC]["sep"]; SEP_D=ENC[BEST_ENC]["d"]
print(f"-> CHON encoder: {BEST_ENC} (sep={SEP:+.3f}, cohens_d={SEP_D:+.3f})")
print("   Cohens_d>=0.8 = tach manh (co cua vuot 247). d<0.5 = yeu -> filter se nhuong ban 247.")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


rn_l2_64    : poison=0.981 streak=0.958 sep=+0.023 cohens_d=+1.700 (np=263 ns=184)
rn_l2_128   : poison=0.990 streak=0.978 sep=+0.013 cohens_d=+1.311 (np=263 ns=184)
rn_l3_64    : poison=0.935 streak=0.880 sep=+0.055 cohens_d=+1.450 (np=263 ns=184)
rn_l3_128   : poison=0.959 streak=0.926 sep=+0.032 cohens_d=+0.977 (np=263 ns=184)
rn_l4_64    : poison=0.758 streak=0.594 sep=+0.164 cohens_d=+1.142 (np=263 ns=184)
rn_l4_128   : poison=0.790 streak=0.650 sep=+0.139 cohens_d=+0.898 (np=263 ns=184)
poisoned    : poison=0.911 streak=0.828 sep=+0.084 cohens_d=+1.371 (np=263 ns=184)
-> CHON encoder: rn_l2_64 (sep=+0.023, cohens_d=+1.700)
   Cohens_d>=0.8 = tach manh (co cua vuot 247). d<0.5 = yeu -> filter se nhuong ban 247.


In [19]:
## 16b-2. BEST_THRESH robust (tai dung logic NB13 cell32): thap nhat trong sai so 3% cua argmin
def _eval_at_thresh(predictor, thr):
    pk={}
    for i,item in enumerate(VAL):
        b,s=predict_image(predictor, item['img']); keep=s>thr
        pk[str(i)]=(b[keep],s[keep]) if len(b) else (b,s)
    return macadd(CLEAN_REF, pk)
_grid=[0.2,0.25,0.3,0.35,0.4,0.45,0.5]
_tv=[(t,_eval_at_thresh(final_pr,t)) for t in _grid]
for _t,_v in _tv: print(f'thr={_t}: val={_v:.4f}')
_vbest=min(v for _,v in _tv); _TOL=0.03
BEST_THRESH=min(t for t,v in _tv if v<=_vbest*(1+_TOL))
print(f'-> BEST_THRESH={BEST_THRESH} (robust, thien giu recall)')


thr=0.2: val=1.0105
thr=0.25: val=0.9413
thr=0.3: val=0.9389
thr=0.35: val=0.9122
thr=0.4: val=0.9558
thr=0.45: val=0.9408
thr=0.5: val=0.9589
-> BEST_THRESH=0.3 (robust, thien giu recall)


In [20]:
## 16c. Calibrate filter — neo T theo top-1 (0.955), khong calibrate mu tren VAL
# cache sim+length cho VAL candidates voi encoder da chon
VAL_SIM=[sim_max_for(VAL_CANDS[i][0], VAL[i]['img'], EMBED_FN, POIS_BANK) for i in range(len(VAL))]
VAL_LEN=[length_of(VAL_CANDS[i][0]) for i in range(len(VAL))]

def apply_filter(cands, sim, L, T, conf_floor, L_rescue):
    b0,s0=cands
    if len(b0)==0: return (b0,s0)
    keep=(sim<T)&(s0>conf_floor)
    if L_rescue>0:                       # rescue box DAI (streak) du sim cao
        keep = keep | ((L>=L_rescue)&(s0>conf_floor))
    return (b0[keep], s0[keep])

def eval_filter(T, conf_floor, L_rescue):
    pk={}; fired=ft=0; kept=st=0; nkeep=0
    for i,item in enumerate(VAL):
        b,s=apply_filter(VAL_CANDS[i], VAL_SIM[i], VAL_LEN[i], T, conf_floor, L_rescue)
        pk[str(i)]=(b,s); nkeep+=len(b)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b) and iou_matrix(np.array([pbx],np.float32),b).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b) and iou_matrix(np.array([sbx],np.float32),b).max()>=MATCH_IOU: kept+=1
    return dict(firing=fired/max(ft,1), retention=kept/max(st,1),
                val=macadd(CLEAN_REF,pk), keep_per_img=nkeep/max(len(VAL),1))

# neo theo top-1: T quanh 0.955-0.96; conf_floor theo BEST_THRESH cua NB13; L_rescue theo phan vi length top-1 (~50)
_allsim=np.concatenate([s for s in VAL_SIM if len(s)]) if any(len(s) for s in VAL_SIM) else np.array([1.0])
T_GRID=sorted(set(np.round(np.percentile(_allsim,[60,70,75,80,85,90,95]),4)))  # thich nghi theo thang cosine cua encoder da chon
print('T_GRID (percentile sim):', T_GRID)
CF_GRID=sorted(set([SUBMIT_THRESH, BEST_THRESH, 0.3]))
LR_GRID=[0.0, 50.0]
rows=[]
for T in T_GRID:
    for cf in CF_GRID:
        for lr in LR_GRID:
            st=eval_filter(T,cf,lr); rows.append((st,T,cf,lr))
rows.sort(key=lambda r:r[0]['val'])
for st,T,cf,lr in rows[:8]:
    print(f'T={T} cf={cf} Lr={lr} | ret={st["retention"]:.3f} fire={st["firing"]:.3f} keep/img={st["keep_per_img"]:.2f} val={st["val"]:.4f}')
# chon: retention>=0.85 (bai hoc: FN phat full-weight) roi val thap nhat
cand=[r for r in rows if r[0]['retention']>=0.85] or rows
BEST_ST,BEST_T,BEST_CF,BEST_LR=cand[0]
print('-> FILTER chon:', {k:round(v,4) for k,v in BEST_ST.items()}, f'| T={BEST_T} conf_floor={BEST_CF} L_rescue={BEST_LR}')

T_GRID (percentile sim): [np.float64(0.979), np.float64(0.9826), np.float64(0.9847), np.float64(0.986), np.float64(0.9878), np.float64(0.9901), np.float64(0.9933)]
T=0.979 cf=0.2 Lr=0.0 | ret=0.767 fire=0.067 keep/img=1.65 val=0.7207
T=0.9826 cf=0.2 Lr=0.0 | ret=0.805 fire=0.123 keep/img=1.84 val=0.7260
T=0.9847 cf=0.2 Lr=0.0 | ret=0.811 fire=0.227 keep/img=2.06 val=0.7872
T=0.9826 cf=0.3 Lr=0.0 | ret=0.553 fire=0.061 keep/img=1.21 val=0.7967
T=0.979 cf=0.3 Lr=0.0 | ret=0.528 fire=0.043 keep/img=1.12 val=0.8115
T=0.9847 cf=0.3 Lr=0.0 | ret=0.560 fire=0.092 keep/img=1.29 val=0.8175
T=0.986 cf=0.3 Lr=0.0 | ret=0.566 fire=0.117 keep/img=1.35 val=0.8433
T=0.986 cf=0.2 Lr=0.0 | ret=0.824 fire=0.307 keep/img=2.25 val=0.8435
-> FILTER chon: {'firing': 0.0675, 'retention': 0.7673, 'val': 0.7207, 'keep_per_img': 1.65} | T=0.979 conf_floor=0.2 L_rescue=0.0


In [21]:
## 16d. Cong an toan: filter chi thay 247 khi VAL xac nhan (retention & val & separation)
BASE_247 = FINAL_STATS['val_macadd']       # val cua ban 247 (khong filter)
FILTER_OK = (SEP_D>=0.8) and (BEST_ST['retention']>=0.85) and (BEST_ST['val']<=BASE_247)
print(f'separation={SEP:+.3f} cohens_d={SEP_D:+.3f} (can d>=0.8)')
print(f'filter val={BEST_ST["val"]:.4f} vs 247 val={BASE_247:.4f} | filter retention={BEST_ST["retention"]:.3f}')
print('=> FILTER_OK =', FILTER_OK, '(True: submission.csv se dung filter; False: giu nguyen ban 247)')

separation=+0.023 cohens_d=+1.700 (can d>=0.8)
filter val=0.7207 vs 247 val=1.0105 | filter retention=0.767
=> FILTER_OK = False (True: submission.csv se dung filter; False: giu nguyen ban 247)


## 17. Submission — ghi 2 ban (247 fallback + top-1 filter), 1 GPU pass

In [22]:
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
def write_sub(pred, path):
    sample=pd.read_csv(SAMPLE_SUB, dtype={'image_id':str}); sub=sample.copy()
    sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))))
    sub['prediction_string']=sub['prediction_string'].fillna(' ')
    sub.loc[sub['prediction_string'].str.strip()=='','prediction_string']=' '
    sub.to_csv(path, index=False)
    tot=sum(len(s) for _,s in pred.values()); return tot

pred_base={}; pred_filt={}
for p in tqdm(test_files, desc='infer+filter test'):
    img=load_image(p); b,s=predict_image(final_pr, img)
    keep=s>BEST_THRESH; pred_base[p.stem]=(b[keep] if len(b) else b, s[keep] if len(b) else s)   # ban 247
    sim=sim_max_for(b, img, EMBED_FN, POIS_BANK); L=length_of(b)
    pred_filt[p.stem]=apply_filter((b,s), sim, L, BEST_T, BEST_CF, BEST_LR)                        # ban top-1 style

t_base=write_sub(pred_base, str(WORK/'submission_247.csv'))
t_filt=write_sub(pred_filt, str(WORK/'submission_filtered.csv'))
# submission.csv mac dinh = ban an toan hon
import shutil
shutil.copy(str(WORK/('submission_filtered.csv' if FILTER_OK else 'submission_247.csv')), SUBMISSION_PATH)
print(f'submission_247.csv      dets={t_base} ({t_base/len(test_files):.3f}/img)')
print(f'submission_filtered.csv dets={t_filt} ({t_filt/len(test_files):.3f}/img)')
print(f'DEFAULT submission.csv = {"FILTERED (top-1 style)" if FILTER_OK else "247 fallback (filter chua dat)"}')

infer+filter test: 100%|██████████| 2000/2000 [04:09<00:00,  8.03it/s]


submission_247.csv      dets=711 (0.355/img)
submission_filtered.csv dets=775 (0.388/img)
DEFAULT submission.csv = 247 fallback (filter chua dat)


In [23]:
## 18. Tong ket
print('='*60)
print('NB21 = de-poison 247 (proven) + filter kieu top-1 (embedding-sim + length + conf).')
print(f'Encoder dung: {BEST_ENC} | separation={SEP:.3f}')
print(f'247 baseline val = {BASE_247:.4f} | filter val = {BEST_ST["val"]:.4f} | FILTER_OK={FILTER_OK}')
print('Quy tac nop:')
print(' - FILTER_OK=True  -> nop submission.csv (da la filtered). Ky vong vuot 247.')
print(' - FILTER_OK=False -> filter chua thang tren VAL; nop submission_247.csv (an toan).')
print(' - Luon giu 1 slot du phong. Khong nop ban kem 247.')

NB21 = de-poison 247 (proven) + filter kieu top-1 (embedding-sim + length + conf).
Encoder dung: rn_l2_64 | separation=0.023
247 baseline val = 1.0105 | filter val = 0.7207 | FILTER_OK=False
Quy tac nop:
 - FILTER_OK=True  -> nop submission.csv (da la filtered). Ky vong vuot 247.
 - FILTER_OK=False -> filter chua thang tren VAL; nop submission_247.csv (an toan).
 - Luon giu 1 slot du phong. Khong nop ban kem 247.
